# MUTCD Multimodal RAG — HPRC (v2)

**What this notebook does:**
1. Extracts every `Figure X-Y` and `Table X-Y` from the MUTCD PDF as a separate cropped PNG (not full pages).
2. Builds three independent indices: section text (MiniLM), figure captions (MiniLM), and figure pixels (CLIP).
3. Hybrid retrieval: exact figure-id match  +  caption lexical match  +  CLIP image-text similarity  +  text section retrieval.
4. Generates the final answer with Qwen2.5-VL (3B by default; flip one constant for 7B).
5. Single `ask("your question")` function — answer printed as Markdown, figure crops shown inline. **No gradio, no proxy, no ports.**

Run cells top to bottom. The first run does the figure extraction and CLIP embedding (a few minutes); subsequent runs hit the cache and are instant.

## 0. Environment
Caches forced into `$SCRATCH/hf_cache` so HOME quota is untouched. Prints the GPU.

In [ ]:
import os, sys, platform
from pathlib import Path

SCRATCH = Path(os.environ.get("SCRATCH", "/tmp"))
BASE_DIR = SCRATCH / "MRAG"
for k, v in (
    ("HF_HOME",            str(SCRATCH / "hf_cache")),
    ("TRANSFORMERS_CACHE", str(SCRATCH / "hf_cache")),
    ("HF_HUB_CACHE",       str(SCRATCH / "hf_cache" / "hub")),
    ("HF_HUB_DISABLE_TELEMETRY", "1"),
    ("TOKENIZERS_PARALLELISM",   "false"),
):
    os.environ.setdefault(k, v)
for k in ("HF_HOME", "TRANSFORMERS_CACHE", "HF_HUB_CACHE"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

import torch
print("python      :", sys.version.split()[0])
print("host        :", platform.node())
print("torch       :", torch.__version__)
print("cuda avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device :", torch.cuda.get_device_name(0))
    print("cuda mem GB :", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
print("SCRATCH     :", SCRATCH)
print("BASE_DIR    :", BASE_DIR, "exists?", BASE_DIR.exists())

## 1. Configuration

All tunable knobs in one place. Most useful ones to play with:
- `VLM_MODEL_NAME` — switch to `"Qwen/Qwen2.5-VL-7B-Instruct"` for noticeably better prose (needs ~17 GB VRAM in bf16; fits an A100/H100, *not* a T4 without 8-bit quant).
- `TOP_K_FIGURES` / `FINAL_FIGURES` — how many figure crops the VLM gets to see.
- `RENDER_DPI` — quality of figure crops. 220 is the sweet spot.

In [ ]:
# ----- Paths -----
PDF_PATH         = BASE_DIR / "mutcd11theditionr1hl.pdf"
SECTIONS_JSON    = BASE_DIR / "mutcd_sections_with_images.json"
PAGE_IMAGE_DIR   = BASE_DIR / "page_images"
FIGURES_DIR      = BASE_DIR / "figures"
FIGURES_JSON     = BASE_DIR / "figures.json"
CACHE_DIR        = BASE_DIR / "mmrag_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PAGE_RECORDS_JSON     = CACHE_DIR / "page_records.json"
SECTION_EMB_NPY       = CACHE_DIR / "section_embeddings.npy"
PAGE_EMB_NPY          = CACHE_DIR / "page_embeddings.npy"
FIG_CAPTION_EMB_NPY   = CACHE_DIR / "figure_caption_embeddings.npy"
FIG_CLIP_EMB_NPY      = CACHE_DIR / "figure_clip_embeddings.npy"

# ----- Models -----
TEXT_EMBED_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
CLIP_EMBED_MODEL_NAME  = "sentence-transformers/clip-ViT-B-32"   # multimodal; same API
VLM_MODEL_NAME         = "Qwen/Qwen2.5-VL-3B-Instruct"
# Upgrade option (uncomment if you have >=24 GB VRAM):
# VLM_MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
VLM_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# ----- Figure extraction -----
RENDER_DPI = 220     # DPI for figure crops
PAGE_DPI   = 180     # DPI for fallback page renders

# ----- Retrieval -----
TOP_K_TEXT      = 8
TOP_K_FIGURES   = 12
FINAL_TEXT      = 4
FINAL_FIGURES   = 4

# Weights for the figure scoring ensemble.
W_CLIP    = 1.00  # image-text similarity
W_CAPTION = 0.80  # caption text similarity (MiniLM)
W_FIG_ID  = 1.50  # bonus when the figure id ("8C-1") is explicit in query
W_KW      = 0.25  # bonus for keyword overlap with caption

# ----- Generation -----
MAX_SECTION_TEXT_CHARS = 1500
MAX_NEW_TOKENS         = 380

print("VLM         :", VLM_MODEL_NAME)
print("CLIP model  :", CLIP_EMBED_MODEL_NAME)
print("figures dir :", FIGURES_DIR)
print("cache dir   :", CACHE_DIR)

## 2. Load section data and (if needed) render page PNGs

Page PNGs are only used as a fallback when a question hits a page that has no extracted figures (e.g. "what does page 1010 show?"). All real figure questions will be answered with figure crops, not page renders.

In [ ]:
import json, re
import fitz
from tqdm.auto import tqdm

with open(SECTIONS_JSON, "r", encoding="utf-8") as f:
    raw_sections = json.load(f)

sections = []
for s in raw_sections:
    page_num = int(s["page_num"])
    sections.append({
        "section":  str(s.get("section", "")).strip(),
        "title":    str(s.get("title", "")).strip(),
        "page_num": page_num,
        "text":     str(s.get("text", "")).strip(),
    })
print("sections loaded:", len(sections))

PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
doc = fitz.open(str(PDF_PATH))
missing = [i for i in range(doc.page_count)
           if not (PAGE_IMAGE_DIR / f"page_{i+1:04d}.png").exists()]
if missing:
    print(f"rendering {len(missing)} missing pages at {PAGE_DPI} DPI ...")
    mat = fitz.Matrix(PAGE_DPI/72.0, PAGE_DPI/72.0)
    for i in tqdm(missing):
        out = PAGE_IMAGE_DIR / f"page_{i+1:04d}.png"
        pix = doc.load_page(i).get_pixmap(matrix=mat, alpha=False)
        pix.save(str(out))
else:
    print("all page PNGs already present.")
print("PDF pages:", doc.page_count)

## 3. Extract figure / table crops from the PDF

Caption-anchored. For every `Figure X-Y` or `Table X-Y` we find in the text, we crop the page region above (for figures) or below (for tables) the caption and save it as a PNG. The result is one image per actual figure, not per page.

Cached. Re-runs are instant once `figures.json` exists.

In [ ]:
CAPTION_RE = re.compile(
    r"^\s*(Figure|Table)\s+"
    r"([A-Z]?\d+[A-Z]?-\d+(?:\([A-Za-z0-9]+\))?[A-Za-z0-9]*)"
    r"\s*[.\u2014:-]?\s*(.{0,250})",
    re.IGNORECASE,
)
SKIP_LINE_RE = re.compile(r"page\s+\d+|chapter\s+\d", re.IGNORECASE)

def _find_captions(page):
    captions = []
    for block in page.get_text("dict").get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            text = " ".join(s["text"] for s in line.get("spans", [])).strip()
            if not text or SKIP_LINE_RE.search(text):
                continue
            m = CAPTION_RE.match(text)
            if not m:
                continue
            captions.append({
                "kind":  m.group(1).title(),
                "id":    m.group(2).upper(),
                "text":  text,
                "title": m.group(3).strip(" .\u2014-:"),
                "bbox":  tuple(line["bbox"]),
            })
    return captions

def _figure_region(caps, i, page_rect):
    bb = caps[i]["bbox"]
    y_top = page_rect.y0 + 30
    for j in range(i - 1, -1, -1):
        if caps[j]["bbox"][3] <= bb[1]:
            y_top = max(y_top, caps[j]["bbox"][3] + 8); break
    return fitz.Rect(page_rect.x0 + 24, y_top, page_rect.x1 - 24, bb[1] - 4)

def _table_region(caps, i, page_rect):
    bb = caps[i]["bbox"]
    y_bot = page_rect.y1 - 30
    for j in range(i + 1, len(caps)):
        if caps[j]["bbox"][1] >= bb[3]:
            y_bot = min(y_bot, caps[j]["bbox"][1] - 8); break
    return fitz.Rect(page_rect.x0 + 24, bb[3] + 4, page_rect.x1 - 24, y_bot)

def _safe(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")

def extract_figures_from_pdf(pdf_path, out_dir, dpi=RENDER_DPI,
                              min_h=60.0, min_w=80.0):
    out_dir.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(str(pdf_path))
    mat = fitz.Matrix(dpi/72.0, dpi/72.0)
    out = []
    for pn in tqdm(range(doc.page_count), desc="extract figures"):
        page = doc.load_page(pn)
        caps = _find_captions(page)
        if not caps: continue
        caps.sort(key=lambda c: c["bbox"][1])
        for i, cap in enumerate(caps):
            rect = _figure_region(caps, i, page.rect) if cap["kind"].lower() == "figure" \
                   else _table_region(caps, i, page.rect)
            if rect.is_empty or rect.height < min_h or rect.width < min_w:
                continue
            pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
            fname = f"{cap['kind'].lower()}_{_safe(cap['id'])}_p{pn+1:04d}.png"
            out_path = out_dir / fname
            pix.save(str(out_path))
            out.append({
                "figure_id":  f"{cap['kind']} {cap['id']}",
                "kind":       cap["kind"],
                "id":         cap["id"],
                "page_num":   pn + 1,
                "caption":    cap["text"],
                "title":      cap["title"],
                "image_path": str(out_path),
                "bbox":       [rect.x0, rect.y0, rect.x1, rect.y1],
                "dpi":        dpi,
            })
    return out

if FIGURES_JSON.exists() and FIGURES_DIR.exists():
    with open(FIGURES_JSON, "r", encoding="utf-8") as f:
        figures = json.load(f)
    print(f"loaded cached figures: {len(figures)}")
else:
    figures = extract_figures_from_pdf(PDF_PATH, FIGURES_DIR, dpi=RENDER_DPI)
    with open(FIGURES_JSON, "w", encoding="utf-8") as f:
        json.dump(figures, f, ensure_ascii=False, indent=2)
    print(f"extracted {len(figures)} figures -> {FIGURES_DIR}")

# Quick distribution
from collections import Counter
print("by kind :", dict(Counter(f["kind"] for f in figures)))
print("example :", figures[0] if figures else "<empty>")

## 4. Page records (lightweight per-page text index, mostly for "what's on page N" questions)

In [ ]:
FIG_RE_PAGE   = re.compile(r"(Figure\s+[A-Za-z0-9.\-]+[^.\n]*)", re.IGNORECASE)
TBL_RE_PAGE   = re.compile(r"(Table\s+[A-Za-z0-9.\-]+[^.\n]*)",  re.IGNORECASE)
SIGN_RE_PAGE  = re.compile(r"\b([A-Z]{1,3}\d{1,2}[A-Za-z0-9\-]*)\b")

def _norm(t): return re.sub(r"\s+", " ", t or "").strip()

if PAGE_RECORDS_JSON.exists():
    with open(PAGE_RECORDS_JSON, "r", encoding="utf-8") as f:
        page_records = json.load(f)
    print("loaded cached page_records:", len(page_records))
else:
    page_records = []
    for i, page in enumerate(tqdm(doc, desc="page records")):
        txt = page.get_text("text") or ""
        page_records.append({
            "page_num":        i + 1,
            "image_path":      str(PAGE_IMAGE_DIR / f"page_{i+1:04d}.png"),
            "page_text":       txt,
            "page_text_norm":  _norm(txt),
            "figure_captions": [_norm(x) for x in FIG_RE_PAGE.findall(txt)[:3]],
            "table_titles":    [_norm(x) for x in TBL_RE_PAGE.findall(txt)[:3]],
            "sign_codes":      sorted(set(SIGN_RE_PAGE.findall(txt)))[:50],
        })
    with open(PAGE_RECORDS_JSON, "w", encoding="utf-8") as f:
        json.dump(page_records, f, ensure_ascii=False)
    print("built and cached page_records:", len(page_records))

## 5. Embeddings — text (MiniLM) + image (CLIP)

Three indices:
1. **Sections** — MiniLM on the section prose. Drives the textual answer.
2. **Figure captions** — MiniLM on each figure's caption. Drives lexical figure recall.
3. **Figure pixels** — CLIP on each figure's PNG. Drives visual figure recall (e.g. "yellow diamond warning sign").

In [ ]:
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer

def _clean(t):
    t = (t or "").lower().replace("\u2013", "-").replace("\u2014", "-")
    return re.sub(r"\s+", " ", t).strip()

# ----- text embed model -----
text_embed = SentenceTransformer(TEXT_EMBED_MODEL_NAME)

section_texts = [
    f"Section {s['section']}. {s['title']}. {s['text'][:2000]}"
    for s in sections
]
if SECTION_EMB_NPY.exists():
    section_embeddings = np.load(SECTION_EMB_NPY)
else:
    section_embeddings = text_embed.encode(section_texts, convert_to_numpy=True,
                                            normalize_embeddings=True,
                                            show_progress_bar=True).astype("float32")
    np.save(SECTION_EMB_NPY, section_embeddings)
print("section_embeddings:", section_embeddings.shape)

page_texts = [
    f"Page {p['page_num']}. "
    f"Figures: {' | '.join(p['figure_captions'])}. "
    f"Tables: {' | '.join(p['table_titles'])}. "
    f"{p['page_text'][:3500]}"
    for p in page_records
]
if PAGE_EMB_NPY.exists():
    page_embeddings = np.load(PAGE_EMB_NPY)
else:
    page_embeddings = text_embed.encode(page_texts, convert_to_numpy=True,
                                         normalize_embeddings=True,
                                         show_progress_bar=True).astype("float32")
    np.save(PAGE_EMB_NPY, page_embeddings)
print("page_embeddings   :", page_embeddings.shape)

# ----- figure caption embeddings (MiniLM) -----
fig_caption_texts = [
    f"{f['figure_id']}. {f['title'] or f['caption']}"
    for f in figures
]
if FIG_CAPTION_EMB_NPY.exists() and len(fig_caption_texts) == np.load(FIG_CAPTION_EMB_NPY).shape[0]:
    fig_caption_embeddings = np.load(FIG_CAPTION_EMB_NPY)
else:
    fig_caption_embeddings = text_embed.encode(fig_caption_texts, convert_to_numpy=True,
                                                normalize_embeddings=True,
                                                show_progress_bar=True).astype("float32")
    np.save(FIG_CAPTION_EMB_NPY, fig_caption_embeddings)
print("fig_caption_embeddings:", fig_caption_embeddings.shape)

# ----- figure CLIP (visual) embeddings -----
clip_model = SentenceTransformer(CLIP_EMBED_MODEL_NAME,
                                 device="cuda" if torch.cuda.is_available() else "cpu")
if FIG_CLIP_EMB_NPY.exists() and len(figures) == np.load(FIG_CLIP_EMB_NPY).shape[0]:
    figure_clip_embeddings = np.load(FIG_CLIP_EMB_NPY)
    print("loaded cached CLIP embeddings.")
else:
    print("encoding figure images with CLIP (this is the slow one) ...")
    images = []
    for f in tqdm(figures, desc="opening figures"):
        try:
            images.append(Image.open(f["image_path"]).convert("RGB"))
        except Exception as e:
            print("skip", f["image_path"], e)
            images.append(Image.new("RGB", (32, 32), "white"))
    figure_clip_embeddings = clip_model.encode(
        images, batch_size=16, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype("float32")
    np.save(FIG_CLIP_EMB_NPY, figure_clip_embeddings)
    # Free memory
    for im in images: im.close()
print("figure_clip_embeddings:", figure_clip_embeddings.shape)

## 6. Retrieval — sections, figures (hybrid), and a tiny fallback for raw-page questions

Figure retrieval combines:
- Direct figure-id parse (`"Figure 8C-1"` in the query => exact bonus)
- CLIP image-text similarity
- MiniLM caption similarity
- Keyword overlap with caption

Weights are in the config cell (`W_CLIP`, `W_CAPTION`, `W_FIG_ID`, `W_KW`).

In [ ]:
FIG_ID_RE  = re.compile(r"\b(figure|table)\s+([A-Z]?\d+[A-Z]?-\d+[A-Za-z0-9]*)\b", re.IGNORECASE)
PAGE_RE    = re.compile(r"\bpage\s+(\d{1,4})\b", re.IGNORECASE)
STOPWORDS  = set("a an the of in on for at to with by and or but is are was were be been being "
                 "this that these those it its as from into about over under between within "
                 "what which where who when how why does do did can could should would will "
                 "mutcd shows show describe explain me".split())

def parse_figure_id(query):
    m = FIG_ID_RE.search(query)
    if not m: return None
    return f"{m.group(1).title()} {m.group(2).upper()}"

def parse_explicit_page(query):
    m = PAGE_RE.search(query or "")
    return int(m.group(1)) if m else None

def _kw(text):
    return {w for w in re.findall(r"[A-Za-z]{4,}", (text or "").lower())
            if w not in STOPWORDS}

def retrieve_sections(query, top_k=TOP_K_TEXT):
    q = text_embed.encode([query], normalize_embeddings=True).astype("float32")[0]
    scores = section_embeddings @ q
    idxs = np.argsort(scores)[::-1][:top_k]
    qn = _clean(query)
    out = []
    for i in idxs:
        s = sections[i]; score = float(scores[i])
        if qn and qn in _clean(s["title"]):           score += 0.20
        if qn and qn in _clean(s["text"][:3000]):     score += 0.10
        out.append({"score": score, **s})
    out.sort(key=lambda x: x["score"], reverse=True)
    return out[:top_k]

def retrieve_figures(query, top_k=TOP_K_FIGURES):
    if not figures:
        return []
    # 1. text-side scores (caption MiniLM)
    qt = text_embed.encode([query], normalize_embeddings=True).astype("float32")[0]
    cap_scores = fig_caption_embeddings @ qt
    # 2. image-side scores (CLIP)
    qv = clip_model.encode([query], normalize_embeddings=True).astype("float32")[0]
    clip_scores = figure_clip_embeddings @ qv
    # 3. combine
    combined = W_CLIP * clip_scores + W_CAPTION * cap_scores
    # 4. figure-id bonus
    explicit_id = parse_figure_id(query)
    if explicit_id:
        wanted = explicit_id.lower()
        for i, f in enumerate(figures):
            if f["figure_id"].lower() == wanted:
                combined[i] += W_FIG_ID
    # 5. keyword overlap with caption
    q_kw = _kw(query)
    if q_kw:
        for i, f in enumerate(figures):
            overlap = len(q_kw & _kw(f["caption"]))
            if overlap:
                combined[i] += W_KW * min(overlap, 3)
    idxs = np.argsort(combined)[::-1][:top_k]
    return [{"score": float(combined[i]),
             "clip_score": float(clip_scores[i]),
             "cap_score":  float(cap_scores[i]),
             **figures[i]}
            for i in idxs]

def retrieve_page_fallback(page_num):
    if 1 <= page_num <= len(page_records):
        return page_records[page_num - 1]
    return None

def fuse_results(query):
    text_hits = retrieve_sections(query, top_k=TOP_K_TEXT)
    fig_hits  = retrieve_figures(query, top_k=TOP_K_FIGURES)
    # If query explicitly asks about a page and we didn't pull figures from it, append a page render.
    explicit_page = parse_explicit_page(query)
    if explicit_page is not None and not any(f["page_num"] == explicit_page for f in fig_hits[:FINAL_FIGURES]):
        pr = retrieve_page_fallback(explicit_page)
        if pr is not None:
            fig_hits.insert(0, {
                "figure_id":  f"Page {explicit_page}",
                "kind":       "Page",
                "id":         str(explicit_page),
                "page_num":   explicit_page,
                "caption":    "(full page render — no figure crop available for this page)",
                "title":      "",
                "image_path": pr["image_path"],
                "score":      999.0,
                "clip_score": 0.0, "cap_score": 0.0,
            })
    return text_hits[:FINAL_TEXT], fig_hits[:FINAL_FIGURES]

print("retrieval ready.")

## 7. Clean retrieved MUTCD text before sending to the VLM

Strips the `Standard 01`, `Option 02`, `Guidance:` prefixes that confuse small models. Collapses whitespace. Keeps numbered list structure.

In [ ]:
PREFIX_RE = re.compile(
    r"^\s*(Standard|Option|Guidance|Support)\s*[:\u2014\-]?\s*\d*\s*[:\u2014\-]?\s*",
    re.IGNORECASE | re.MULTILINE,
)
ITEM_RE = re.compile(r"^(\d{1,2})\.\s+", re.MULTILINE)

def clean_mutcd_text(text):
    if not text: return ""
    t = PREFIX_RE.sub("", text)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

# Smoke test on the first retrieved section, only if non-empty.
if sections:
    sample = clean_mutcd_text(sections[0]["text"])[:400]
    print("cleaned sample:\n---\n" + sample + "\n---")

## 8. Load the VLM

Default Qwen2.5-VL-3B. To upgrade prose quality, change `VLM_MODEL_NAME` in the config cell to `"Qwen/Qwen2.5-VL-7B-Instruct"` and re-run from there. The 7B is *much* more fluent and follows instructions far better, but needs ~17 GB VRAM in bf16.

In [ ]:
vlm_model = vlm_processor = vlm_pipe = None
_vlm_mode = None
try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    from qwen_vl_utils import process_vision_info  # noqa: F401
    vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VLM_MODEL_NAME, torch_dtype=VLM_DTYPE, device_map="auto",
    )
    vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_NAME)
    _vlm_mode = "explicit"
    print("VLM loaded (explicit):", VLM_MODEL_NAME)
except Exception as e:
    print("explicit loader failed, falling back to pipeline:", repr(e))
    from transformers import pipeline
    vlm_pipe = pipeline("image-text-to-text", model=VLM_MODEL_NAME,
                        device_map="auto", torch_dtype=VLM_DTYPE)
    _vlm_mode = "pipeline"
    print("VLM loaded (pipeline):", VLM_MODEL_NAME)

## 9. Prompt + generate

Shorter, focused prompt with a fixed output structure. The model gets:
- The figure crops (real figures, not full pages)
- Cleaned section text
- A list of figure ids it's allowed to cite

In [ ]:
def build_vlm_messages(question, text_hits, fig_hits):
    content = []
    # Attach each figure crop as an image.
    used_figs = []
    for f in fig_hits:
        ip = f.get("image_path", "")
        if not ip or not Path(ip).exists():
            continue
        used_figs.append(f)
        if _vlm_mode == "explicit":
            content.append({"type": "image", "image": f"file://{ip}"})
        else:
            content.append({"type": "image", "image": Image.open(ip)})

    # Build textual evidence.
    fig_lines = []
    for i, f in enumerate(used_figs, 1):
        cap = f.get("caption") or f.get("figure_id", "")
        fig_lines.append(f"[Image {i}] {f['figure_id']} (page {f['page_num']}): {cap}")

    text_lines = []
    for r in text_hits:
        body = clean_mutcd_text(r["text"])[:MAX_SECTION_TEXT_CHARS]
        text_lines.append(
            f"[Section {r['section']} \u2014 {r['title']} (p.{r['page_num']})]\n{body}"
        )

    allowed_cites = (
        "Allowed citations:\n"
        + "\n".join(f"  - Section {r['section']} (p.{r['page_num']})" for r in text_hits)
        + "\n"
        + "\n".join(f"  - {f['figure_id']} (p.{f['page_num']})" for f in used_figs)
    )

    prompt = (
        "You are an expert reader of the Manual on Uniform Traffic Control "
        "Devices (MUTCD). Answer the user's question using ONLY the evidence "
        "below (text excerpts and the attached figure images, which are listed "
        "in order as [Image 1], [Image 2], ...).\n\n"
        "Style requirements:\n"
        "- 3 to 6 sentences of clear, professional prose. No bullet points unless "
        "the question explicitly asks for a list.\n"
        "- Quote exact MUTCD wording when defining a Standard provision.\n"
        "- If a figure is relevant, describe what the figure shows in one sentence "
        "and reference it as e.g. 'Figure 8C-1'.\n"
        "- Never invent figure numbers, section numbers, or page numbers.\n"
        "- End with a 'Citations:' line listing the sections and figures you used, "
        "chosen ONLY from the allowed list below.\n\n"
        f"Question: {question}\n\n"
        f"Figure evidence:\n" + ("\n".join(fig_lines) if fig_lines else "(none)") + "\n\n"
        f"Text evidence:\n" + "\n\n".join(text_lines) + "\n\n"
        f"{allowed_cites}\n\n"
        "Now answer."
    )
    content.append({"type": "text", "text": prompt})
    return [{"role": "user", "content": content}], used_figs

def ask_vlm(question, text_hits, fig_hits):
    messages, used_figs = build_vlm_messages(question, text_hits, fig_hits)
    if _vlm_mode == "explicit":
        from qwen_vl_utils import process_vision_info
        text = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = vlm_processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt",
        ).to(vlm_model.device)
        with torch.inference_mode():
            gen = vlm_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, gen)]
        out_text = vlm_processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        return out_text.strip(), used_figs
    else:
        output = vlm_pipe(text=messages, max_new_tokens=MAX_NEW_TOKENS,
                          do_sample=False, return_full_text=False)
        gen = output[0]["generated_text"]
        if isinstance(gen, list):
            gen = gen[-1]["content"]
        return str(gen).strip(), used_figs

print("VLM prompt+generate ready.")

## 10. The `ask()` function — your daily interface

Type `ask("your question")` in any cell. The answer prints as Markdown, and the actual figure crops the model used are shown inline below it.

Returns the dict for further inspection if you want to debug retrieval scores.

In [ ]:
from IPython.display import display, Markdown, Image as IPImage

def ask(question, show_scores=False, show_text=False, max_fig_width=620):
    text_hits, fig_hits = fuse_results(question)
    try:
        answer, used_figs = ask_vlm(question, text_hits, fig_hits)
    except Exception as e:
        print("[VLM error]", repr(e))
        answer = "(VLM failed — see error above)"
        used_figs = fig_hits

    display(Markdown(f"### Q\n{question}\n\n### Answer\n{answer}"))

    if used_figs:
        display(Markdown("### Figures the model saw"))
        for i, f in enumerate(used_figs, 1):
            ip = f.get("image_path", "")
            if ip and Path(ip).exists():
                display(Markdown(
                    f"**[Image {i}]** {f['figure_id']} \u2014 page {f['page_num']}  \n"
                    f"*{f.get('caption','')}*"
                    + (f"  \nscore={f.get('score',0):.3f} "
                       f"(clip={f.get('clip_score',0):.3f}, cap={f.get('cap_score',0):.3f})"
                       if show_scores else "")
                ))
                display(IPImage(filename=ip, width=max_fig_width))

    if show_text:
        display(Markdown("### Retrieved section context"))
        for r in text_hits:
            display(Markdown(
                f"**Section {r['section']} \u2014 {r['title']}** (p.{r['page_num']}) "
                f"\u00b7 score={r['score']:.3f}\n\n"
                f"{clean_mutcd_text(r['text'])[:600]}..."
            ))

    return {
        "question": question,
        "answer":   answer,
        "text_hits": text_hits,
        "fig_hits":  used_figs,
    }

print("ask() ready. Try:  ask('Explain Figure 8C-1')")

## 11. Try it

In [ ]:
_ = ask("Explain Figure 8C-1", show_scores=True)

In [ ]:
_ = ask("What does MUTCD require for pedestrian hybrid beacons?", show_text=True)

In [ ]:
_ = ask("Show me the warning signs and plaques for grade crossings.")

## 12. Debugging retrieval (no VLM call)

Use this cell to test the retriever in isolation. Cheap. No GPU.

In [ ]:
def show_retrieval(query, k_text=4, k_fig=6):
    text_hits, fig_hits = fuse_results(query)
    print("QUERY:", query)
    print("-- text --")
    for r in text_hits[:k_text]:
        print(f"  Sec {r['section']:>8s} | p.{r['page_num']:>4d} | s={r['score']:.3f} | {r['title'][:60]}")
    print("-- figures --")
    for f in fig_hits[:k_fig]:
        print(f"  {f['figure_id']:<22s} | p.{f['page_num']:>4d} | "
              f"s={f.get('score',0):.3f} (clip={f.get('clip_score',0):.3f}, "
              f"cap={f.get('cap_score',0):.3f}) | {f.get('title','')[:60]}")

for q in [
    "Explain Figure 8C-1",
    "yellow diamond warning sign at a horizontal curve",
    "pedestrian hybrid beacon signal sequence",
    "What does page 1010 show?",
]:
    show_retrieval(q)
    print()